# Generating Analytical Kinematic Models

This notebook introduces the workflow for generating analytical kinematic models in RobotBlockSet. Its purpose is to show how to build kinematic models from Denavit-Hartenberg (DH) parameters, URDF descriptions, or MJCF models.


## What this notebook covers

The examples below demonstrate the main approaches for model generation: defining a robot through DH parameters, extracting model information from URDF files, and using MJCF-based descriptions available in the RobotBlockSet workflow.

Use this notebook as a practical reference when you want to generate symbolic or analytical kinematic models that can be reused for kinematics, dynamics-related utilities, or custom robot definitions.


# Imports


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import sympy as sp
import os

from robotblockset.tools import get_rbs_path
from robotblockset.transformations import p2t, rot_x, rot_y, rot_z
from robotblockset.gen_models import gen_kinmodel_dh, gen_kinmodel_dh_all, gen_kinmodel_urdf, show_urdf, show_urdf_tree, gen_gravmodel_urdf, show_mjcf_tree, show_mjcf, gen_kinmodel_mjcf, gen_gravmodel_mjcf
from robotblockset.robot_models import kinmodel_lwr
from robotblockset.multi_robots import join_robot, robot_reverse, join_robot_fixed

np.set_printoptions(precision=4, suppress=True)

# Initialization

In [ ]:
URDF_PATH = get_rbs_path() + "/urdf_models/"
MJCF_PATH = get_rbs_path() + "/mujoco/mjcf_models/"
MODEL_FILE = "tutorial_models.py"
if os.path.exists(MODEL_FILE):
    os.remove(MODEL_FILE)

# Generation of kinematic models

To be able to control the motion of a robot we use in RBS kinematic models of robot manipulators. Kinematic models can be generated using the Denavit-Hartenberg (DH) notation, or the model is imported from the URDF file describing the robot, or the model is imported from the MuJoCO MJCF file describing the robot.


## DH parameters

Using DH-parameters we have to define the structure (as a function or a variable) with required fields

In [ ]:
panda = {
    "name": "panda",
    "description": "FRANKA-Emika Panda",
    "nj": 7,
    "a": [0, 0, 0.0825, -0.0825, 0, 0.088, 0],
    "alpha": [-sp.pi / 2, sp.pi / 2, sp.pi / 2, -sp.pi / 2, sp.pi / 2, sp.pi / 2, 0],
    "d": [0.333, 0, 0.316, 0, 0.384, 0, 0.107],
    "theta": [0] * 7,
}



Then, using the provided function `gen_kinmodel_dh` the function `kinmodel_<robot.name>` for the direct kinematics in symbolic form is generated.



In [ ]:
gen_kinmodel_dh(panda, filename=MODEL_FILE)

Kinematic model for 'panda' using DH parameters has been generated in tutorial_models.py.


## URDF

If an URDF description file for a robot exists, we can use function `gen_kinmodel_urdf` to generate the kinematic model. In this case, we can generate a model for only part of the serial kinematic chain, For that, we have to select the initial and the final link in the kinematic chain. 

In [ ]:
urdf_file = URDF_PATH + "panda/panda.urdf"



To show the robot structure given in the URDF file we can use



In [ ]:
show_urdf_tree(urdf_file)

world
  |- panda_world_joint [fixed] -> panda_link0
    panda_link0
      |- panda_joint1 [revolute] -> panda_link1
        panda_link1
          |- panda_joint2 [revolute] -> panda_link2
            panda_link2
              |- panda_joint3 [revolute] -> panda_link3
                panda_link3
                  |- panda_joint4 [revolute] -> panda_link4
                    panda_link4
                      |- panda_joint5 [revolute] -> panda_link5
                        panda_link5
                          |- panda_joint6 [revolute] -> panda_link6
                            panda_link6
                              |- panda_joint7 [revolute] -> panda_link7
                                panda_link7
                                  |- panda_joint8 [fixed] -> panda_link8
                                    panda_link8
                                      |- panda_hand_joint [fixed] -> panda_hand
                                        panda_hand
                                    

or

In [ ]:
show_urdf(urdf_file)

Joint         Type                                     Name                                   Parent                                    Child
------- ---------- ---------------------------------------- ---------------------------------------- ----------------------------------------
    1       fixed                        panda_world_joint                                    world                              panda_link0
    2    revolute                             panda_joint1                              panda_link0                              panda_link1
    3    revolute                             panda_joint2                              panda_link1                              panda_link2
    4    revolute                             panda_joint3                              panda_link2                              panda_link3
    5    revolute                             panda_joint4                              panda_link3                              panda_link4
    6    re

We generate the model defining the initial and final link of the kinematic chaina.

In [ ]:
gen_kinmodel_urdf("panda_urdf", urdf_file, description="Panda URDF", initial_link="world", final_link="panda_link8", prefix="", filename=MODEL_FILE)

Kinematic model for 'panda_urdf' has been generated in tutorial_models.py.


## MuJoCo MJCF

If an MJCF model file for a scene with robot exists, we can use function `gen_kinmodel_mjcf` to generate the kinematic model. In this case, we can generate a model for only part of the serial kinematic chain, For that, we have to select the initial and the final link in the kinematic chain. 

> ℹ️ **Note:** The MJCF adapter is built for the serial-arm style models used in this repo. It handles pos, quat, euler, axisangle, fixed bodies, hinge joints, and slide joints, but not the full MJCF feature set.

In [ ]:
mjcf_file = MJCF_PATH + "panda_scene.xml"

To show the robot structure given in the MJCF file we can use

In [ ]:
show_mjcf_tree(mjcf_file, start_body="panda")

panda
    panda_link0
        panda_link1
          |- panda_joint1 [hinge]
            panda_link2
              |- panda_joint2 [hinge]
                panda_link3
                  |- panda_joint3 [hinge]
                    panda_link4
                      |- panda_joint4 [hinge]
                        panda_link5
                          |- panda_joint5 [hinge]
                            panda_link6
                              |- panda_joint6 [hinge]
                                panda_link7
                                  |- panda_joint7 [hinge]
                                    panda_flange
                                        panda_hand
                                            panda_TCP
                                            panda_left_finger
                                              |- panda_finger_joint1 [hinge]
                                            panda_right_finger
                                              |- panda_finger_joint2 [hinge]

or

In [ ]:
show_mjcf(mjcf_file)

Joint         Type                                     Name                                   Parent                                    Child
------- ---------- ---------------------------------------- ---------------------------------------- ----------------------------------------
    1       fixed                          world_to_Target                                    world                                   Target
    2       fixed                           world_to_panda                                    world                                    panda
    3       fixed                     panda_to_panda_link0                                    panda                              panda_link0
    4       hinge                             panda_joint1                              panda_link0                              panda_link1
    5       hinge                             panda_joint2                              panda_link1                              panda_link2
    6      

We generate the model defining the initial and final link of the kinematic chaina.

In [ ]:
gen_kinmodel_mjcf("panda_mjcf", mjcf_file, description="Panda MJCF", initial_link="world", final_link="panda_flange", prefix="", filename=MODEL_FILE)

Kinematic model for 'panda_mjcf' from MJCF has been generated in tutorial_models.py.


## Verify kinematic models

The generated kinematic model function has a standard from and is used as this
```python
x, J = kinmodel_<robot.name>(q)
```
Optionally we can define the tool-center-point (tcp). 
```python
x, J = kinmodel_<robot.name>(q, tcp=[0, 0, 0.2])
```

In [ ]:
from tutorial_models import kinmodel_panda, kinmodel_panda_urdf, kinmodel_panda_mjcf

In [ ]:
q = np.random.randn(7)
print(q)
TCP =  np.array([[0.7071, 0.7071, 0.0, 0.0], [-0.7071, 0.7071, 0.0, 0.0], [0.0, 0.0, 1, 0.1034], [0.0, 0.0, 0.0, 1.0]])

[-1.4542 -0.5212 -1.4837 -1.9011 -1.759  -1.4588  0.5846]


In [ ]:
x1, J1 = kinmodel_panda(q, tcp=TCP)
x2, J2 = kinmodel_panda_urdf(q, tcp=TCP)
x3, J3 = kinmodel_panda_mjcf(q, tcp=TCP)
print(x1)
print(x2)
print(x3)
print(J2 - J1)
print(J3 - J1)


[-0.3276 -0.1001  0.6798  0.8066  0.5577  0.1047  0.1654]
[-0.3276 -0.1001  0.6798  0.8066  0.5577  0.1047  0.1654]
[-0.3276 -0.1001  0.6798  0.8066  0.5577  0.1047  0.1654]
[[ 0. -0.  0. -0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0. -0. -0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]]
[[ 0. -0.  0. -0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0. -0. -0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]]


# Joining serial kinematic chains

The Toolbox provides functions to join serial kinematic chains into one kinematic chain.

| Function                                        | Description                                               |
| ----------------------------------------------- | --------------------------------------------------------- |
| `x, J = join_robot(x1, R1, J1, x2, R2, J2)`         | Joining two robot kinematics                              |
| `x, J = join_reverse_robot(x1, R1, J1, x2, R2, J2)` | Joining two robot kinematics, first in reverse order      |
| `x, J = join_robot_reverse(x1, R1, J1, x2, R2, J2)` | Joining two robot kinematics, second in reverse order     |
| `x, J = join_robot_fixed(x1, R1, J1, x2, R2)`      | Join kinematic model with fixed kinematic structure       |
| `x, J = join_fixed_robot(x1, R1, x2, R2, J2)`      | Join fixed kinematic structure with robot kinematic model |
| `x, J = robot_reverse(x1, R1, J1)`               | Robot kinematics from end-effector to base                |

For example, if we want to joint two KUKA LWR robot arms to a dual-arm robot which starts in the end-effector of the first robot ($\mathcal{S}b$) and ends at the end-effector of the second robot ($\mathcal{S}t$).

<img src="assets/LWR_sim.PNG" style="zoom: 33%;" />

then we get the kinematic model of joined dual-arm robot as



In [ ]:
q1 = np.array([-0.3460, 0.1384, -0.2517, -1.6195, 0.5419, 1.1927, -0.3092])
tcp1 = p2t([0, 0, 0.2], out="T")
xb1 = np.array([0, -0.045 / 2 - 0.054, 0.92])
Rb1 = rot_y(35 / 180 * np.pi, out="R") @ rot_x(60 / 180 * np.pi, out="R") @ rot_z(-120 / 180 * np.pi, out="R")
q2 = np.array([0.0828, 1.0148, -0.1233, -1.3104, -0.4346, 0.4044, 1.5402])
tcp2 = p2t([0, 0, 0.2], out="T")
xb2 = np.array([0, 0.045 / 2 + 0.054, 0.92])
Rb2 = rot_y(35 / 180 * np.pi, out="R") @ rot_x(-60 / 180 * np.pi, out="R") @ rot_z(120 / 180 * np.pi, out="R")
xb = Rb1.T @ (xb2 - xb1)
Rb = Rb1.T @ Rb2
p1, R1, J1 = kinmodel_lwr(q1, tcp1, out="pR")
p1r, R1r, J1r = robot_reverse(p1, R1, J1)
p1f, R1f, J1f = join_robot_fixed(p1r, R1r, J1r, xb, Rb)
p2, R2, J2 = kinmodel_lwr(q2, tcp2, out="pR")
pp, RR, JJ = join_robot(p1f, R1f, J1f, p2, R2, J2)
print("Bimanual robot position:\n", pp)
print("Bimanual robot orinetation:\n", RR)
print("Bimanual robot Jacobian matrix:\n", JJ)

Bimanual robot position:
 [ 0.0001 -0.125  -0.0001]
Bimanual robot orinetation:
 [[ 0.0008 -1.     -0.0001]
 [-1.     -0.0008 -0.    ]
 [ 0.      0.0001 -1.    ]]
Bimanual robot Jacobian matrix:
 [[ 0.1938  0.0201  0.1842 -0.3617 -0.1247  0.2648 -0.125   0.3972 -0.4688
   0.2765  0.569   0.0033 -0.2779  0.    ]
 [ 0.4704 -0.2097  0.4558  0.1321  0.246   0.0846 -0.0001  0.6128  0.3771
   0.586  -0.2889  0.1093  0.0083  0.    ]
 [-0.1583 -0.4927 -0.1349  0.3007  0.1107  0.0379 -0.      0.1583 -0.517
   0.0624  0.1391 -0.      0.      0.    ]
 [-0.2868  0.1489 -0.4146  0.0793 -0.8853 -0.3043 -0.      0.2868  0.39
   0.8946 -0.4139  0.3932  0.0298 -0.0001]
 [ 0.4096  0.9121  0.4084 -0.874  -0.2828  0.9526  0.     -0.4096  0.8756
  -0.4337 -0.8951 -0.0117  0.9996 -0.    ]
 [ 0.866  -0.3821  0.8132  0.4793 -0.3692 -0.     -1.      0.866   0.2849
   0.1081 -0.1656 -0.9194 -0.     -1.    ]]


# Gravitational models

Using URDF definition of a robot we can also generate the gravitational model of a robot using funnction `gen_kinmodel_urdf`.

In [ ]:
gen_gravmodel_urdf("panda_urdf", urdf_file, description="Panda URDF", initial_link="world", final_link="panda_link8", prefix="", filename=MODEL_FILE)

Gravity model for 'panda_urdf' has been generated in tutorial_models.py.


or using MJCF model we can generate the gravitational model of a robot using funnction `gen_kinmodel_mjcf`.

>ℹ️**Note:** Function `gen_kinmodel_mjcf` by default considers also robot link masses. To get tha same model as when using `gen:gravmodel_urdf` the argument `load_only=True` has to be used

In [ ]:
gen_gravmodel_mjcf("panda_mjcf", mjcf_file, description="Panda MJCF", initial_link="world", final_link="panda_flange", prefix="", filename=MODEL_FILE, load_only=True)

Gravity model for 'panda_mjcf' from MJCF has been generated in tutorial_models.py.


## Verify gravitational models

In [ ]:
from tutorial_models import gravmodel_panda_urdf, gravmodel_panda_mjcf

In [ ]:
q = np.random.randn(7)
print(q)
load = 1

[-0.0828  0.1573 -0.3743 -0.7407 -0.0914 -1.4305 -0.1682]


In [ ]:
g1 = gravmodel_panda_urdf(q, load=1)
g2 = gravmodel_panda_mjcf(q, load=1)
print(g1)
print(g2)

[ 0.     -2.3073 -0.0681  1.0534  0.1173 -1.346   0.    ]
[ 0.     -2.3073 -0.0681  1.0534  0.1173 -1.346   0.    ]
